# EDA - LFW (Many People, Few Images)

Notebook nay dung de xem nhanh dataset LFW theo preset `100 x 10`; kiem tra threshold, split va hien thi anh mau sau MTCNN.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

ROOT

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from src.process import analyze_subject_count_thresholds, process_lfw_many_people_few_images_dataset
from src.utils import plot_sample_images

In [ ]:
config = {
    "dataset_name": "lfw",
    "role": "many_people_few_images",
    "test_size": 0.2,
    "random_state": 42,
    "face_detection": "mtcnn",
    "min_face_area_ratio": 0.08,
    "sample_grid_count": 15,
    "thresholds": [5, 10, 15, 20, 25, 30],
}

config

## Threshold overview

In [ ]:
threshold_df = pd.DataFrame(
    analyze_subject_count_thresholds(
        config["dataset_name"],
        thresholds=config["thresholds"],
        face_detection=config["face_detection"],
        min_face_area_ratio=config["min_face_area_ratio"],
    )
)
display(threshold_df)

## Process dataset va luu artifact da tach train/test

In [ ]:
processed = process_lfw_many_people_few_images_dataset(
    flatten=False,
    test_size=config["test_size"],
    random_state=config["random_state"],
    save_artifacts=True,
)

summary = processed["summary"]
display(pd.Series({
    "dataset_name": summary["dataset_name"],
    "role": config["role"],
    "samples_total": summary["samples_total"],
    "classes_total": summary["classes_total"],
    "train_shape": summary["train_shape"],
    "test_shape": summary["test_shape"],
    "stratify_used": summary["stratify_used"],
    "processed_dir": processed["output_dir"],
}))

In [ ]:
subject_counts = pd.Series(processed["metadata"]["subject_names"], name="subject_name").value_counts().sort_values(ascending=False)
subject_count_hist = subject_counts.value_counts().sort_index().rename_axis("images_per_subject").reset_index(name="num_subjects")

display(subject_counts.describe())
display(subject_counts.head(20).rename("images"))
display(subject_count_hist)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(subject_count_hist["images_per_subject"].astype(str), subject_count_hist["num_subjects"], color="#2f6db2")
ax.set_xlabel("Images per subject")
ax.set_ylabel("Number of subjects")
ax.set_title("LFW subject-count distribution (100 x 10 preset)")
ax.grid(axis="y", alpha=0.2)
plt.tight_layout()

In [ ]:
train_counts = pd.Series(processed["train_metadata"]["subject_names"], name="train").value_counts().sort_index()
test_counts = pd.Series(processed["test_metadata"]["subject_names"], name="test").value_counts().sort_index()
split_counts = pd.concat([train_counts, test_counts], axis=1).fillna(0).astype(int)

display(split_counts.head(20))

fig, ax = plt.subplots(figsize=(14, 4.5))
split_counts.head(25).plot(kind="bar", ax=ax, width=0.85)
ax.set_xlabel("Subject")
ax.set_ylabel("Images")
ax.set_title("Train/Test counts for first 25 subjects")
ax.grid(axis="y", alpha=0.2)
plt.tight_layout()

## Hien thi anh

In [ ]:
plot_sample_images(
    processed["X_train"],
    labels=processed["y_train"],
    n_samples=config["sample_grid_count"],
)

In [ ]:
plot_sample_images(
    processed["X_test"],
    labels=processed["y_test"],
    n_samples=min(10, processed["X_test"].shape[0]),
)

In [ ]:
train_subject_names = processed["train_metadata"]["subject_names"]
representative_indices = []
seen_subjects = set()
for index, subject_name in enumerate(train_subject_names):
    if subject_name in seen_subjects:
        continue
    seen_subjects.add(subject_name)
    representative_indices.append(index)
    if len(representative_indices) >= config["sample_grid_count"]:
        break

representative_labels = np.asarray([train_subject_names[index] for index in representative_indices], dtype=object)
plot_sample_images(
    processed["X_train"][representative_indices],
    labels=representative_labels,
    n_samples=len(representative_indices),
)